# Network Parameters

## Introduction

A `Network`'s parameters, like `s`, `z` or `y`, are numpy arrays that know which `Network` they belong to. Each has its scalar components, like `db` or `deg`, as attributes, and both can plot themselves:

* **Network**
    * **parameter**: `s`, `z`, `y`, `t`, `a`, `g`, `h`
        * **component**: `re`, `im`, `mag`, `db`, `db10`, `deg`, `rad`, `deg_unwrap`, `vswr`, `time_db`, ...

So `ntwk.s.db.plot()` plots the magnitude of the s-parameters in dB, and `ntwk.s.db` holds those values. The flat names you may know, like `ntwk.s_db` and `ntwk.plot_s_db()`, still work and give the same results.

In [ ]:
%matplotlib inline

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

import skrf as rf

rf.stylely()
ring_slot = rf.data.ring_slot
ring_slot

## Parameters are arrays

`ring_slot.s` is a complex array of shape `fxnxn`, with frequency first, just as it always was. It prints as one, and it goes into numpy functions as one.

In [ ]:
ring_slot.s[:2]

In [ ]:
isinstance(ring_slot.s, np.ndarray), ring_slot.s.shape

What is new is its type, `NetworkParameter`, which adds the components and the plot methods.

In [ ]:
type(ring_slot.s)

## Plotting a parameter

`plot()` draws the s-parameters on a Smith chart. The `m` and `n` arguments pick a single parameter, starting from 0.

In [ ]:
ring_slot.s.plot()

In [ ]:
ring_slot.s.plot(m=0, n=0)

The other parameters are drawn in the complex plane. Both can also be drawn in polar coordinates with `plot_polar()`, or in the complex plane with `plot_complex()`.

In [ ]:
ring_slot.z.plot()
plt.axis('equal');

In [ ]:
ring_slot.s.plot_polar()

## Components

Each parameter has its scalar components as attributes. Each one is an array too, of type `Projection`, which plots itself against frequency.

In [ ]:
ring_slot.s.db[:2]

In [ ]:
ring_slot.s.db.plot()

In [ ]:
ring_slot.s.deg.plot(m=1, n=0)

They take the same arguments as the flat plot methods, like `ax`, so they combine into larger figures.

In [ ]:
fig, (ax_db, ax_deg) = plt.subplots(1, 2, figsize=(10, 4))
ring_slot.s.db.plot(m=1, n=0, ax=ax_db)
ring_slot.s.deg_unwrap.plot(m=1, n=0, ax=ax_deg)
fig.tight_layout()

The component names come from the same table as the flat attributes, so each nested name has a flat twin that holds the same values:

| nested | flat |
|---|---|
| `ntwk.s.db` | `ntwk.s_db` |
| `ntwk.s.db.plot(m=1, n=0)` | `ntwk.plot_s_db(m=1, n=0)` |
| `ntwk.z.im` | `ntwk.z_im` |
| `ntwk.s.plot()` | `ntwk.plot_s_smith()` |
| `ntwk.z.plot()` | `ntwk.plot_z_complex()` |

In [ ]:
np.array_equal(ring_slot.s.db, ring_slot.s_db)

## Still numpy

Numpy methods work on parameters and components as usual, which makes it easy to mark values on a plot.

In [ ]:
ring_slot.s.db.plot(m=0, n=0)
plt.axhline(ring_slot.s.db[:, 0, 0].min(), color='k', linestyle='--');

Anything numpy makes from them, a slice, a sum or a product, is a plain `numpy.ndarray` again, so you can pass the results anywhere an array is expected.

In [ ]:
type(ring_slot.s[:, 0, 0]), type(ring_slot.s * 2), type(np.linalg.inv(ring_slot.z))

## Combining with Networks

Since every `Network` has them, they combine with frequency slicing and with picking a single s-parameter as a one-port.

In [ ]:
ring_slot['80-90ghz'].s.db.plot()

In [ ]:
ring_slot.s21.s.deg_unwrap.plot()

## Time domain

The time components transform the parameter to the time domain. Window the network first to tame the band edges, as with the flat methods.

In [ ]:
rf.data.ring_slot_meas.windowed().s.time_db.plot()

The step and impulse responses need a transform with its own options, so they stay flat methods, like `ntwk.plot_s_time_step()` or `ntwk.plot_z_time_step()`.

## Changing values

`ntwk.s` is a view of the s-matrix the `Network` stores, so writing into it changes the network.

In [ ]:
ntwk = ring_slot.copy()
ntwk.name = 'halved S21'
ntwk.s[:, 1, 0] *= 0.5

ring_slot.s.db.plot(m=1, n=0)
ntwk.s.db.plot(m=1, n=0)

The other parameters are computed from the s-matrix every time you read them, so writing into them does not change the network. Assign a whole new matrix instead, `ntwk.z = new_z`.